# Time Series Track · Time Series Foundations & Classical Methods

**Track:** Traditional AI Domains — Time Series  
**Prerequisites:** ML/01 (Math Foundations), NB03b (EDA)  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

### 🔍 Socratic Deep Check
*By the end of this notebook, you will be able to answer:*  
**Why can't you just use linear regression on time series data — and when CAN a simple ARIMA model outperform a Transformer?**

---

## Why Time Series Matters for AI Engineers

Time series is everywhere in AI production systems:

| Domain | Time Series Task | Business Impact |
|--------|-----------------|----------------|
| **Finance** | Stock/revenue forecasting | Revenue planning, risk management |
| **IoT / Manufacturing** | Sensor anomaly detection | Predictive maintenance, safety |
| **MLOps** | Latency/error rate monitoring | SLA compliance, capacity planning |
| **E-commerce** | Demand forecasting | Inventory optimization, pricing |
| **Energy** | Load/renewable output forecasting | Grid stability, cost optimization |

```
Main Curriculum Connections:
  NB07 (Kafka/Streaming)  → Where time series DATA arrives
  NB08 (Feature Store)    → Where time series FEATURES are served
  NB21 (Monitoring)       → Time series of MODEL METRICS
  NB21b (Observability)   → Time series of SYSTEM METRICS (Prometheus)
```

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors treat time series like tabular data — flatten everything, throw it into XGBoost, and wonder why test accuracy is 95% but production predictions are garbage. Seniors understand temporal ordering, stationarity, and data leakage from the future. They choose ARIMA for simple univariate forecasting and save deep learning for complex multivariate problems.

**Mental Model:** Time series is like a river. You can only stand at one point and look UPSTREAM (past). You cannot look DOWNSTREAM (future). Any model that "accidentally" sees downstream data during training will appear to work perfectly but fail completely when deployed.

**Common Junior Pitfall:** Using random train/test split on time series data. This leaks future information into training (e.g., training on Jan, Mar, May and testing on Feb, Apr — the model has already "seen" data from after Feb). Always use a **temporal split**: train on past, test on future.

---

## 📑 Table of Contents

  * [🔍 Socratic Deep Check](#socratic-deep-check)
* [Why Time Series Matters for AI Engineers](#why-time-series-matters-for-ai-engineers)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Time Series Fundamentals](#1-time-series-fundamentals)
  * [Stationarity: The Most Important Concept](#stationarity-the-most-important-concept)
* [2 · ARIMA: The Classical Workhorse](#2-arima-the-classical-workhorse)
* [3 · Exponential Smoothing & Prophet](#3-exponential-smoothing-prophet)
* [4 · Temporal Train/Test Splits & Cross-Validation](#4-temporal-traintest-splits-cross-validation)
* [Knowledge Check](#knowledge-check)
  * [Q1: Why can't you use random train/test split on time series?](#q1-why-cant-you-use-random-traintest-split-on-time-series)
  * [Q2: What does the "I" in ARIMA do?](#q2-what-does-the-i-in-arima-do)
  * [Q3: When does ARIMA beat deep learning?](#q3-when-does-arima-beat-deep-learning)
* [Practical Practice](#practical-practice)
  * [Exercise 1](#exercise-1)
  * [Exercise 2](#exercise-2)


In [ ]:
!pip install -q pandas numpy matplotlib statsmodels scikit-learn

In [ ]:
# Cell 1 — Generate realistic time series data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 365 * 3  # 3 years of daily data
dates = pd.date_range('2022-01-01', periods=n, freq='D')

# Components of a real time series:
trend = np.linspace(100, 180, n)                           # Upward trend
seasonality = 20 * np.sin(2 * np.pi * np.arange(n) / 365) # Yearly cycle
weekly = 5 * np.sin(2 * np.pi * np.arange(n) / 7)         # Weekly cycle
noise = np.random.normal(0, 5, n)                          # Random noise

# Add a structural break (e.g., product launch)
break_effect = np.where(np.arange(n) > 600, 30, 0)

values = trend + seasonality + weekly + noise + break_effect
ts = pd.Series(values, index=dates, name='daily_revenue')

print(f'Time Series: {len(ts)} daily observations')
print(f'Date range: {ts.index[0].date()} to {ts.index[-1].date()}')
print(f'\nStatistics:')
print(ts.describe().round(1).to_string())

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Time Series Decomposition', fontsize=14, fontweight='bold')
axes[0,0].plot(dates, values, linewidth=0.8); axes[0,0].set_title('Full Series')
axes[0,1].plot(dates, trend + break_effect, color='red'); axes[0,1].set_title('Trend + Structural Break')
axes[1,0].plot(dates[:90], seasonality[:90] + weekly[:90], color='green'); axes[1,0].set_title('Seasonality (90 days)')
axes[1,1].hist(noise, bins=40, color='gray', alpha=0.7); axes[1,1].set_title('Noise Distribution')
plt.tight_layout()
plt.show()

---
## 1 · Time Series Fundamentals

### Stationarity: The Most Important Concept

A time series is **stationary** if its statistical properties (mean, variance) don't change over time.

| | Stationary | Non-Stationary |
|---|---|---|
| Mean | Constant | Changes over time (trend) |
| Variance | Constant | Changes over time |
| ARIMA works? | ✅ Yes | ❌ Must difference first |
| Test | ADF test (p < 0.05 = stationary) | ADF test (p > 0.05) |

**Why it matters:** ARIMA and many classical models REQUIRE stationarity. You achieve it by **differencing** (subtracting each value from the previous one).

In [ ]:
# Cell 2 — Stationarity testing and differencing
from statsmodels.tsa.stattools import adfuller

def test_stationarity(series, name=''):
    result = adfuller(series.dropna())
    print(f'{name:25s} ADF Statistic: {result[0]:.4f}, p-value: {result[1]:.6f}'
          f' → {"✅ Stationary" if result[1] < 0.05 else "❌ Non-Stationary"}')

test_stationarity(ts, 'Original series')
test_stationarity(ts.diff().dropna(), 'After 1st differencing')

print('\nDifferencing removes the trend, making the series stationary.')
print('ARIMA\'s "I" (Integrated) does this automatically: d=1 means 1 round of differencing.')

---
## 2 · ARIMA: The Classical Workhorse

ARIMA(p, d, q):
- **AR(p):** AutoRegressive — use past **p** values to predict
- **I(d):** Integrated — difference **d** times for stationarity
- **MA(q):** Moving Average — use past **q** forecast errors

SARIMA(p,d,q)(P,D,Q,s) adds seasonal components with period **s** (e.g., s=7 for weekly, s=365 for yearly).

In [ ]:
# Cell 3 — ARIMA/SARIMA forecasting
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Temporal split (NEVER random split for time series!)
train = ts[:'2024-06-30']
test = ts['2024-07-01':]
print(f'Train: {len(train)} days ({train.index[0].date()} to {train.index[-1].date()})')
print(f'Test:  {len(test)} days ({test.index[0].date()} to {test.index[-1].date()})\n')

# Fit ARIMA
model = ARIMA(train, order=(5, 1, 2))
fitted = model.fit()
print(fitted.summary().tables[1])

# Forecast
forecast = fitted.forecast(steps=len(test))
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
print(f'\nForecast Results:')
print(f'  MAE:  {mae:.2f} (average error in revenue units)')
print(f'  RMSE: {rmse:.2f}')

# Plot
plt.figure(figsize=(14, 5))
plt.plot(train.index[-90:], train[-90:], label='Train', color='steelblue')
plt.plot(test.index, test, label='Actual', color='black', linewidth=1.5)
plt.plot(test.index, forecast, label=f'ARIMA Forecast (MAE={mae:.1f})', color='coral', linestyle='--')
plt.title('ARIMA(5,1,2) Forecast'); plt.legend(); plt.show()

---
## 3 · Exponential Smoothing & Prophet

| Method | Best For | Handles Seasonality | External Regressors |
|--------|---------|--------------------|-----------------|
| ARIMA | General univariate | Via SARIMA | No |
| ETS (Holt-Winters) | Strong seasonal patterns | Yes (additive/multiplicative) | No |
| Prophet | Business time series with holidays | Yes + holiday effects | Yes |

In [ ]:
# Cell 4 — Exponential Smoothing (ETS)
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Holt-Winters with additive seasonality
hw_model = ExponentialSmoothing(
    train, 
    seasonal_periods=7,      # Weekly seasonality
    trend='add',             # Additive trend
    seasonal='add',          # Additive seasonality
).fit()

hw_forecast = hw_model.forecast(len(test))
hw_mae = mean_absolute_error(test, hw_forecast)
print(f'Holt-Winters MAE: {hw_mae:.2f}')
print(f'ARIMA MAE:        {mae:.2f}')
print(f'\nWinner: {"Holt-Winters" if hw_mae < mae else "ARIMA"}')

# Prophet template (requires pip install prophet)
print('\n--- Prophet Usage Template ---')
print('''
from prophet import Prophet

df = pd.DataFrame({'ds': dates, 'y': values})  # Prophet requires 'ds' and 'y'
model = Prophet(
    changepoint_prior_scale=0.05,  # Trend flexibility
    seasonality_mode='additive',
)
model.add_country_holidays(country_name='US')  # Built-in holiday effects
model.fit(df_train)

future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)
model.plot_components(forecast)  # Trend + weekly + yearly + holidays
''')

---
## 4 · Temporal Train/Test Splits & Cross-Validation

**NEVER use random train/test split on time series.**

```
WRONG (random split — data leakage!):
  ██░░██░░██░░██░░    ██ = train, ░░ = test
  The model sees Jan, Mar, May (train) and predicts Feb, Apr (test)
  → It's "predicting" the PAST with knowledge of the FUTURE

RIGHT (temporal split):
  ████████████░░░░    Train on past, test on future

BEST (expanding window cross-validation):
  Fold 1: ████░░░░░░░░
  Fold 2: ██████░░░░░░
  Fold 3: ████████░░░░
  Fold 4: ██████████░░
```

In [ ]:
# Cell 5 — Time series cross-validation
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
print('=== Expanding Window Cross-Validation ===')
for i, (train_idx, test_idx) in enumerate(tscv.split(ts)):
    train_dates = ts.index[train_idx]
    test_dates = ts.index[test_idx]
    print(f'  Fold {i+1}: Train {train_dates[0].date()} → {train_dates[-1].date()} '
          f'({len(train_idx)} days) | '
          f'Test {test_dates[0].date()} → {test_dates[-1].date()} ({len(test_idx)} days)')

print('\nKey: train window EXPANDS each fold. Test is always AFTER train.')

---
## Knowledge Check

### Q1: Why can't you use random train/test split on time series?
<details><summary>Answer</summary>

Random splitting creates **temporal leakage** — the model trains on future data and tests on past data. It will appear accurate but fail completely in production where you can only predict forward in time. Always use temporal splits.
</details>

### Q2: What does the "I" in ARIMA do?
<details><summary>Answer</summary>

**Integrated (d)** means differencing the series d times to achieve stationarity. If d=1, each value is replaced by the difference from the previous value (y_t - y_{t-1}), removing linear trends.
</details>

### Q3: When does ARIMA beat deep learning?
<details><summary>Answer</summary>

ARIMA excels on **univariate, short-history, well-structured** series (< 1000 points, clear seasonality). Deep learning needs large datasets across many related series to learn useful patterns. For a single store's weekly revenue, ARIMA/ETS wins.
</details>

---
## Practical Practice

### Exercise 1
Use `auto_arima` from `pmdarima` to automatically find the best (p,d,q) for the dataset above. Compare its MAE with the manual ARIMA(5,1,2).

### Exercise 2
Fit a SARIMA model with weekly seasonality (s=7). Does it improve over plain ARIMA?

---
**Next →** `02_ml_for_time_series.ipynb` — Feature engineering and tree-based models for time series